In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("sample_to_run_info.csv")

/var/folders/vr/sdqsty_d49bg8cdvl7kw6vz80000gn/T/ipykernel_13014/2515147133.py:1: DtypeWarning: Columns (22,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sample_to_run_info.csv")


In [3]:
thresh = int(len(df) * 0.5)          # minimum non-null count to keep a column
df = df.dropna(axis=1, thresh=thresh)

In [4]:
df = df.drop(['more', 'collection_date', 'longitude', 'latitude', 'sample_name', 'original_sample_description', 'sample_id', 'second_sample_id', 'disease', 'instrument_model'], axis=1)

In [5]:
# Check whether certain diseases are overrepresented on a given platform for instrument_model.
df

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country
0,PRJDB3418,DRR028772,AMPLICON,3000.0,Healthy,Japan
1,PRJDB3418,DRR028773,AMPLICON,3000.0,Healthy,Japan
2,PRJDB3418,DRR028774,AMPLICON,3000.0,Healthy,Japan
3,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan
4,PRJDB3418,DRR028776,AMPLICON,3000.0,Healthy,Japan
...,...,...,...,...,...,...
118860,PRJNA556801,SRR9851941,AMPLICON,51320.0,"Hepatitis, Autoimmune",China
118861,PRJNA556801,SRR9851942,AMPLICON,45711.0,Healthy,China
118862,PRJNA556801,SRR9851943,AMPLICON,60044.0,"Hepatitis, Autoimmune",China
118863,PRJNA556801,SRR9851944,AMPLICON,45185.0,"Hepatitis, Autoimmune",China


In [6]:
df['phenotype'].value_counts()

phenotype
Healthy                        48241
Colorectal Neoplasms            5543
Crohn Disease                   3516
COVID-19                        2911
Parkinson Disease               2169
                               ...  
Colonic Polyps                     5
Urolithiasis                       5
Colitis, Ischemic                  4
Dry Eye Syndromes                  4
Meibomian Gland Dysfunction        3
Name: count, Length: 292, dtype: int64

In [7]:
# phenotype_counts = df['phenotype'].value_counts()
# valid_phenos = phenotype_counts[phenotype_counts >= 1000].index
# df = df[df['phenotype'].isin(valid_phenos)].copy()

keep_phenos = [
    "Healthy",
#     "Colorectal Neoplasms",
    "Crohn Disease",
    "Colitis, Ulcerative",
]

df = df[df["phenotype"].isin(keep_phenos)].copy()

In [8]:
df['country'].value_counts()

country
China                       14709
United States of America    13830
Japan                        5712
Denmark                      2600
Netherlands                  1521
Finland                      1491
Spain                        1301
Italy                        1237
Germany                      1121
Canada                       1006
India                         998
Australia                     859
South Korea                   829
Sweden                        756
United Kingdom                535
Ghana                         528
Belgium                       461
Ireland                       443
Colombia                      415
France                        366
Brazil                        266
Russia                        214
Thailand                      181
Malaysia                      161
Poland                        153
Czech Republic                142
Indonesia                     141
Philippines                   136
Israel                        132
Singap

In [9]:
df['experiment_type'].value_counts()

experiment_type
AMPLICON        37533
Metagenomics    16087
Name: count, dtype: int64

In [10]:
df

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country
0,PRJDB3418,DRR028772,AMPLICON,3000.0,Healthy,Japan
1,PRJDB3418,DRR028773,AMPLICON,3000.0,Healthy,Japan
2,PRJDB3418,DRR028774,AMPLICON,3000.0,Healthy,Japan
3,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan
4,PRJDB3418,DRR028776,AMPLICON,3000.0,Healthy,Japan
...,...,...,...,...,...,...
118848,PRJNA556801,SRR9851929,AMPLICON,42288.0,Healthy,China
118856,PRJNA556801,SRR9851937,AMPLICON,34177.0,Healthy,China
118857,PRJNA556801,SRR9851938,AMPLICON,39966.0,Healthy,China
118858,PRJNA556801,SRR9851939,AMPLICON,45325.0,Healthy,China


In [11]:
abundance = pd.read_csv('species_abundance.csv')

In [12]:
abundance

,id,loaded_uid,ncbi_taxon_id,taxon_rank_level,relative_abundance,accession_id
0,1,81104,-1,genus,1.951900,DRR358335
1,2,81104,544,genus,1.074570,DRR358335
2,3,81104,561,genus,0.849570,DRR358335
3,4,81104,570,genus,0.062180,DRR358335
4,5,81104,816,genus,23.296990,DRR358335
...,...,...,...,...,...,...
5541266,5541267,774,1980681,genus,0.286862,SRR9951896
5541267,5541268,774,2039302,genus,0.755403,SRR9951896
5541268,5541269,774,2316020,genus,1.415185,SRR9951896
5541269,5541270,774,2742598,genus,0.984892,SRR9951896


In [13]:
abundance['taxon_rank_level'].value_counts()

taxon_rank_level
genus      2780064
species    2761207
Name: count, dtype: int64

In [14]:
# Keep only genus level
abund_genus = abundance[abundance["taxon_rank_level"] == "genus"].copy()

# Drop rows with ncbi_taxon_id = -1 (unassigned/other)
abund_genus = abund_genus[abund_genus["ncbi_taxon_id"] != -1].copy()

In [15]:
# Pivot: run_id x ncbi_taxon_id, values = relative_abundance
abund_wide = (
    abund_genus
    .pivot_table(index="accession_id",      # this matches run_id in df
                 columns="ncbi_taxon_id",
                 values="relative_abundance",
                 aggfunc="sum",
                 fill_value=0.0)
    .reset_index()
)

# Make column names nicer (e.g., taxon_544, taxon_561, ...)
abund_wide.columns = [
    "run_id" if c == "accession_id" else f"taxon_{int(c)}"
    for c in abund_wide.columns
]

In [16]:
abundance

,id,loaded_uid,ncbi_taxon_id,taxon_rank_level,relative_abundance,accession_id
0,1,81104,-1,genus,1.951900,DRR358335
1,2,81104,544,genus,1.074570,DRR358335
2,3,81104,561,genus,0.849570,DRR358335
3,4,81104,570,genus,0.062180,DRR358335
4,5,81104,816,genus,23.296990,DRR358335
...,...,...,...,...,...,...
5541266,5541267,774,1980681,genus,0.286862,SRR9951896
5541267,5541268,774,2039302,genus,0.755403,SRR9951896
5541268,5541269,774,2316020,genus,1.415185,SRR9951896
5541269,5541270,774,2742598,genus,0.984892,SRR9951896


In [17]:
merged = df.merge(abund_wide, on="run_id", how="inner")

In [18]:
merged

,project_id,run_id,experiment_type,nr_reads_sequenced,phenotype,country,taxon_6,taxon_10,taxon_16,taxon_18,...,taxon_3090883,taxon_3109381,taxon_3119832,taxon_3120720,taxon_3120725,taxon_3120754,taxon_3121626,taxon_3277338,taxon_3373482,taxon_3409995
0,PRJDB3418,DRR028775,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,PRJDB3418,DRR028783,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,PRJDB3418,DRR028785,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,PRJDB3418,DRR028787,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,PRJDB3418,DRR028789,AMPLICON,3000.0,Healthy,Japan,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31201,PRJNA556801,SRR9851929,AMPLICON,42288.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31202,PRJNA556801,SRR9851937,AMPLICON,34177.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31203,PRJNA556801,SRR9851938,AMPLICON,39966.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31204,PRJNA556801,SRR9851939,AMPLICON,45325.0,Healthy,China,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [38]:
# Keep only the three phenotypes of interest
keep_phenos = ["Healthy", "Crohn Disease", "Colitis, Ulcerative"]
df3 = merged[merged["phenotype"].isin(keep_phenos)].copy()

# Map phenotype -> integer label
label_map = {
    "Healthy": 0,
    "Crohn Disease": 1,
    "Colitis, Ulcerative": 2,
}
df3["label"] = df3["phenotype"].map(label_map)

In [39]:
# Features = all taxon_* columns
taxon_cols = [c for c in df3.columns if c.startswith("taxon_")]

X = df3[taxon_cols].values
y = df3["label"].values  # values in {0,1,2}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [40]:
# Multinomial L1-penalized logistic regression (LASSO-style)
lasso_multi = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(
        penalty="l1",
        solver="saga",
        multi_class="multinomial",
        C=0.1,               # tune this: smaller -> more sparsity
        class_weight="balanced",
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    ))
])

In [ ]:
lasso_multi.fit(X_train, y_train)

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
